<a href="https://colab.research.google.com/github/sokrypton/ColabDesign/blob/v1.1.1/af/examples/peptide_binder_design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AfDesign - peptide binder design
For a given protein target and protein binder length, generate/hallucinate a protein binder sequence AlphaFold thinks will bind to the target structure. To do this, we maximize number of contacts at the interface and maximize pLDDT of the binder.

**WARNING**
1.   This notebook is in active development and was designed for demonstration purposes only.
2.   Using AfDesign as the only "loss" function for design might be a bad idea, you may find adversarial sequences (aka. sequences that trick AlphaFold).

In [1]:
#@title **setup**
import os
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os
from colabdesign import mk_afdesign_model, clear_mem
from colabdesign.shared.utils import copy_dict
from colabdesign.af.alphafold.common import residue_constants

from IPython.display import HTML
import numpy as np

In [2]:
#@title **prep inputs**
import re
#@markdown ---
#@markdown **target info**
pdb = "/media/sda/anthony/colabdesign_modified/test_scripts/all5_iga.pdb" #@param {type:"string"}
#@markdown - enter PDB code or UniProt code (to fetch AlphaFoldDB model) or leave blink to upload your own
target_chain = "A" #@param {type:"string"}
target_hotspot = "" #@param {type:"string"}
if target_hotspot == "": target_hotspot = None
#@markdown - restrict loss to predefined positions on target (eg. "1-10,12,15")
target_flexible = False #@param {type:"boolean"}
#@markdown - if defined, will initialize design with this sequence

binder_chain = "" #@param {type:"string"}
#@markdown - if defined, supervised loss is used (binder_len is ignored)
fix_pos = "B1-B4,B7,B11,B15,B18-B19,B22-B25,B28,B31-B33,B35-B36,B38-B62"
#@markdown ---
#@markdown **model config**
use_multimer = False #@param {type:"boolean"}
#@markdown - use alphafold-multimer for design
num_recycles = 0 #@param ["0", "1", "3", "6"] {type:"raw"}
num_models = "2" #@param ["1", "2", "3", "4", "5", "all"]
num_models = 5 if num_models == "all" else int(num_models)
#@markdown - number of trained models to use during optimization


x = {"pdb_filename":pdb,
     "chain":target_chain,
     "binder_len":62,
     "binder_chain":"B",
     "hotspot":target_hotspot,
     "use_multimer":use_multimer,
     "rm_target_seq":target_flexible,
     "fix_pos":fix_pos}
         
#if "x_prev" not in dir() or x != x_prev:
clear_mem()
model = mk_afdesign_model(protocol="binder",
                          use_multimer=x["use_multimer"],
                          num_recycles=num_recycles,
                          recycle_mode="sample",
                          data_dir="/media/sda/anthony/colabdesign_modified/params",
                          )
model.prep_inputs(**x,
                  ignore_missing=False)
x_prev = copy_dict(x)
print("target length:", model._target_len)
print("binder length:", model._binder_len)
binder_len = model._binder_len

model.opt["weights"]["ptmenergy"] = 0.5
model.opt["weights"]["dgram_cce"] = 0.00
model.opt["weights"]["plddt"] = 0.05

print("weights",model.opt["weights"])


target length: 214
binder length: 62
weights {'con': 0.0, 'dgram_cce': 0.0, 'exp_res': 0.0, 'fape': 0.0, 'helix': 0.0, 'i_con': 0.0, 'i_pae': 0.0, 'pae': 0.0, 'plddt': 0.05, 'rmsd': 0.0, 'seq_ent': 0.0, 'ptmenergy': 0.5}


In [3]:
#@title **run AfDesign**
from scipy.special import softmax

optimizer = "logits" #@param ["pssm_semigreedy", "3stage", "semigreedy", "pssm", "logits", "soft", "hard"]
#@markdown - `pssm_semigreedy` - uses the designed PSSM to bias semigreedy opt. (Recommended)
#@markdown - `3stage` - gradient based optimization (GD) (logits → soft → hard)
#@markdown - `pssm` - GD optimize (logits → soft) to get a sequence profile (PSSM).
#@markdown - `semigreedy` - tries X random mutations, accepts those that decrease loss
#@markdown - `logits` - GD optimize logits inputs (continious)
#@markdown - `soft` - GD optimize softmax(logits) inputs (probabilities)
#@markdown - `hard` - GD optimize one_hot(logits) inputs (discrete)

#@markdown WARNING: The output sequence from `pssm`,`logits`,`soft` is not one_hot. To get a valid sequence use the other optimizers, or redesign the output backbone with another protocol like ProteinMPNN.

#@markdown ----
#@markdown #### advanced GD settings
GD_method = "sgd" #@param ["adabelief", "adafactor", "adagrad", "adam", "adamw", "fromage", "lamb", "lars", "noisy_sgd", "dpsgd", "radam", "rmsprop", "sgd", "sm3", "yogi"]
learning_rate = 0.1 #@param {type:"raw"}
norm_seq_grad = True #@param {type:"boolean"}
dropout = True #@param {type:"boolean"}

#model.restart(seq=binder_seq)
model.set_optimizer(optimizer=GD_method,
                    learning_rate=learning_rate,
                    norm_seq_grad=norm_seq_grad)
models = model._model_names[:num_models]

flags = {"num_recycles":num_recycles,
         "models":models,
         "dropout":dropout}

if optimizer == "3stage":
  model.design_3stage(10, 5, 1, **flags)
  pssm = softmax(model._tmp["seq_logits"],-1)

if optimizer == "pssm_semigreedy":
  model.design_pssm_semigreedy(100, 30, **flags)
  pssm = softmax(model._tmp["seq_logits"],1)

if optimizer == "semigreedy":
  model.design_pssm_semigreedy(0, 32, **flags)
  pssm = None

if optimizer == "pssm":
  model.design_logits(120, e_soft=1.0, num_models=1, ramp_recycles=True, **flags)
  model.design_soft(32, num_models=1, **flags)
  flags.update({"dropout":False,"save_best":True})
  model.design_soft(10, num_models=num_models, **flags)
  pssm = softmax(model.aux["seq"]["logits"],-1)

O = {"logits":model.design_logits,
     "soft":model.design_soft,
     "hard":model.design_hard}

if optimizer in O:
  O[optimizer](120, num_models=1, ramp_recycles=True, **flags)
  flags.update({"dropout":False,"save_best":True})
  O[optimizer](10, num_models=num_models, **flags)
  pssm = softmax(model.aux["seq"]["logits"],-1)

model.save_pdb(f"{model.protocol}.pdb")

Gradients:
seq: shape=(1, 62, 20), mean=0.000187, std=0.011728
1 models [0] recycles 0 hard 0 soft 0 temp 1 seqid 0.76 loss -2.00 plddt 0.89 ptm 0.68 i_ptm 0.09 ptmenergy -4.01 rmsd 32.60
Gradients:
seq: shape=(1, 62, 20), mean=0.001192, std=0.013948
2 models [0] recycles 0 hard 0 soft 0 temp 1 seqid 0.71 loss -2.02 plddt 0.90 ptm 0.68 i_ptm 0.10 ptmenergy -4.04 rmsd 45.44
Gradients:
seq: shape=(1, 62, 20), mean=-0.000812, std=0.008927
3 models [0] recycles 0 hard 0 soft 0 temp 1 seqid 0.76 loss -2.04 plddt 0.88 ptm 0.68 i_ptm 0.09 ptmenergy -4.10 rmsd 23.01
Gradients:
seq: shape=(1, 62, 20), mean=0.000214, std=0.004897
4 models [0] recycles 0 hard 0 soft 0 temp 1 seqid 0.77 loss -2.09 plddt 0.87 ptm 0.68 i_ptm 0.09 ptmenergy -4.20 rmsd 48.29
Gradients:
seq: shape=(1, 62, 20), mean=0.000303, std=0.006669
5 models [1] recycles 0 hard 0 soft 0 temp 1 seqid 0.79 loss -2.41 plddt 0.86 ptm 0.68 i_ptm 0.07 ptmenergy -4.83 rmsd 57.91
Gradients:
seq: shape=(1, 62, 20), mean=-0.000437, std=0.00

KeyboardInterrupt: 

In [ ]:
print(model.aux["log"])

{'con': 1.9729523658752441, 'dgram_cce': 353.50506591796875, 'exp_res': 0.07046575844287872, 'fape': 223.778564453125, 'i_con': 5.252229690551758, 'i_pae': 0.8169946670532227, 'pae': 0.6629838943481445, 'plddt': 0.8351053595542908, 'rmsd': 50.35124206542969, 'seq_ent': 0.20575526356697083, 'loss': 353.50506591796875, 'i_ptm': 0.1420828402042389, 'ptm': 0.696556806564331, 'ptmenergy': -3.6472315788269043, 'hard': 1.0, 'soft': 1.0, 'temp': 0.01, 'seqid': 0.7258064516129032, 'recycles': 0, 'models': [0, 1]}


In [ ]:
#@title display hallucinated protein {run: "auto"}
color = "pLDDT" #@param ["chain", "pLDDT", "rainbow"]
show_sidechains = False #@param {type:"boolean"}
show_mainchains = False #@param {type:"boolean"}
color_HP = False #@param {type:"boolean"}
animate = True #@param {type:"boolean"}
model.plot_pdb(show_sidechains=show_sidechains,
               show_mainchains=show_mainchains,
               color=color, color_HP=color_HP, animate=animate)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
HTML(model.animate(dpi=100))

In [ ]:
model.save_pdb(f"{model.protocol}.pdb")
model.get_seqs()

['KEVKCDELEDAQQEAIEELWRLNLSDLQILFYIQKIHDAKTVEGVKALLEEALKLNDAKKEI']

In [ ]:
#@markdown ### Amino acid probabilties
import plotly.express as px
alphabet = "ACDEFGHIKLMNPQRSTVWY"
if "pssm" in dir() and pssm is not None:
  fig = px.imshow(pssm.mean(0).T,
                  labels=dict(x="positions", y="amino acids", color="probability"),
                  y=residue_constants.restypes,
                  zmin=0,
                  zmax=1,
                  template="simple_white",
                )
  fig.update_xaxes(side="top")
  fig.show()

ModuleNotFoundError: No module named 'plotly'

In [ ]:
# log
model._tmp["best"]["aux"]["log"]